# Nano PSM Data Upload And Training

Use this notebook after local dataset generation has passed schema, runtime, and quality gates. Upload only gated datasets to Hugging Face Hub.

In [ ]:
!pip install -U huggingface_hub datasets safetensors onnx onnxruntime
# Install torch separately if the Colab runtime does not already provide the desired CUDA build.
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
HF_DATASET_REPO = 'chkrishna2001/nano-psm'
HF_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-checkpoints'
DATASET_REVISION = 'main'

LOCAL_DATA_DIR = '/content/psm-data'
LOCAL_CHECKPOINT_DIR = '/content/nano-psm-checkpoints'
REPO_DIR = '/content/PSM'

## Upload Gated Dataset

Run this only if you manually uploaded or mounted a locally generated dataset folder into Colab. The folder must contain `train.jsonl`, `validation.jsonl`, `metadata.json`, and report files from the quality gate.

In [ ]:
from huggingface_hub import upload_folder

GATED_LOCAL_FOLDER = '/content/gated-psm-data'  # change if mounted elsewhere

# Uncomment after confirming this folder is from a passed local gate.
# upload_folder(
#     repo_id=HF_DATASET_REPO,
#     repo_type='dataset',
#     folder_path=GATED_LOCAL_FOLDER,
#     path_in_repo='.',
#     commit_message='upload gated nano psm training data'
# )

## Download Dataset And Code

In [ ]:
from huggingface_hub import snapshot_download

dataset_dir = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    revision=DATASET_REVISION,
    local_dir=LOCAL_DATA_DIR,
    resume_download=True,
)
print(dataset_dir)

In [ ]:
!git clone https://github.com/chkrishna2001/PSM.git {REPO_DIR} || true
%cd {REPO_DIR}
!git pull

## Resume Checkpoints

In [ ]:
from huggingface_hub import snapshot_download

try:
    checkpoint_dir = snapshot_download(
        repo_id=HF_CHECKPOINT_REPO,
        repo_type='model',
        local_dir=LOCAL_CHECKPOINT_DIR,
        resume_download=True,
    )
except Exception as exc:
    print('No checkpoint snapshot yet:', exc)
    checkpoint_dir = LOCAL_CHECKPOINT_DIR

print(checkpoint_dir)

## Train Debug Config First

Start with a short run. Increase `MAX_STEPS` only after metrics and checkpoint upload work.

In [ ]:
MAX_STEPS = 500
EVAL_EVERY = 100
SAVE_EVERY = 100

!python nano-psm/src/nano_psm/train.py \
  --config nano-psm/configs/debug-4m.json \
  --train {LOCAL_DATA_DIR}/train.jsonl \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint-dir {LOCAL_CHECKPOINT_DIR} \
  --resume auto \
  --device auto \
  --max-steps {MAX_STEPS} \
  --eval-every {EVAL_EVERY} \
  --save-every {SAVE_EVERY}


## Evaluate Best Checkpoint


In [ ]:
!python nano-psm/src/nano_psm/evaluate.py \
  --config nano-psm/configs/debug-4m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto


## Inspect Validation Mistakes

Run this after evaluation. If this returns an empty list, rerun with `--show-correct` to sample successful predictions.

In [ ]:
!python nano-psm/src/nano_psm/inspect_predictions.py \
  --config nano-psm/configs/debug-4m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto \
  --limit 30


## Upload Checkpoints

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(
    repo_id=HF_CHECKPOINT_REPO,
    repo_type='model',
    private=True,
    exist_ok=True,
)

upload_folder(
    repo_id=HF_CHECKPOINT_REPO,
    repo_type='model',
    folder_path=LOCAL_CHECKPOINT_DIR,
    path_in_repo='.',
    commit_message='sync nano psm checkpoint'
)